# Practice 4 · Regularization in Code

| Marker | Meaning |
|---|---|
| 🟢 EDIT | you write this part |
| 🟡 READ | read it, no need to change |
| ⚪ RUN | just run it as it is |

### I. Background

One regularizer is cheap enough to use every time: **data augmentation** — the model never sees the
same picture twice. In this practice you switch it on and off, then write the flip yourself and
plug it in.

### II. Task Requirements

1. **Q1** — write *make_train_loader(augment, batch_size)*.
2. **Q2** — write your own *RandomHorizontalFlip(p)* working on tensors.
3. Run the cells and compare the numbers with the table below.

Steps: open the notebook in Haima, write the 🟢 parts, run the cells from top to bottom.

### III. Input and Output

| name | meaning | expected |
|---|---|---|
| *make_train_loader(False)* | no augmentation | deterministic: the same image always gives the same tensor |
| *make_train_loader(True)* | with augmentation | random: the same image gives different tensors |
| batch | one batch | *(B, 3, 32, 32)*, normalised (mean ≈ 0, std ≈ 1) |
| *RandomHorizontalFlip(0)* | p = 0 | the input unchanged |
| *RandomHorizontalFlip(1)* | p = 1 | *torch.flip(x, [-1])* |
| *RandomHorizontalFlip(0.5)* | p = 0.5 | sometimes unchanged, sometimes flipped |

### IV. Who does what

- **Q1**: required for undergraduate students (postgraduates do it as well).
- **Q2**: required for postgraduate students; optional for undergraduates.
  An undergraduate who also finishes Q2 **gets bonus marks**; leaving it unfinished
  **costs nothing**.


## Question 1 · Augmentation on and off

Write *make_train_loader(augment, batch_size=64)* returning a *DataLoader* over the first
*N_SUB* training images:

- *augment=False*: *ToTensor()* → *Normalize(MEAN, STD)* only — deterministic.
- *augment=True*: *RandomCrop(32, padding=4)* → *RandomHorizontalFlip()* → *ToTensor()* →
  *Normalize(MEAN, STD)*.

Same subset, same shuffle, only the transform differs.


In [1]:
# 🟢 EDIT: build the loader in make_train_loader() and run the cell.
# 🟡 READ: train_raw, N_SUB, MEAN and STD are given above.
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import CIFAR10

torch.manual_seed(0)

# 🟡 READ: CIFAR10 ships with the assignment; ./data holds cifar-10-batches-py
DATA_ROOT = "data"
if not os.path.exists(os.path.join(DATA_ROOT, "cifar-10-batches-py")):
    raise FileNotFoundError(
        "CIFAR-10 not found under ./data - please contact the teaching assistant "
        "and copy the folder cifar-10-batches-py next to this notebook."
    )

MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2470, 0.2435, 0.2613)
train_raw = CIFAR10(DATA_ROOT, train=True, download=False, transform=None)
N_SUB = 2000
print("raw training images:", len(train_raw), "| used in this practice:", N_SUB)


def make_train_loader(augment, batch_size=64):
    """DataLoader over the first N_SUB images, with or without augmentation."""
    # 🟢 CODE HERE - 3 parts:
    #   1) build tf with transforms.Compose:
    #        augment=True  -> RandomCrop(32, padding=4), RandomHorizontalFlip(),
    #                         ToTensor(), Normalize(MEAN, STD)
    #        augment=False -> ToTensor(), Normalize(MEAN, STD)          (deterministic)
    #   2) base = CIFAR10(DATA_ROOT, train=True, download=False, transform=tf)
    #      ds   = torch.utils.data.Subset(base, range(N_SUB))
    #   3) return DataLoader(ds, batch_size=batch_size, shuffle=True)
    if augment:
        tf = transforms.Compose([
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(MEAN, STD),
        ])
    else:
        tf = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(MEAN, STD),
        ])
    base = CIFAR10(DATA_ROOT, train=True, download=False, transform=tf)
    ds = torch.utils.data.Subset(base, range(N_SUB))
    return DataLoader(ds, batch_size=batch_size, shuffle=True)


# ⚪ RUN: the same image twice, with and without augmentation
for flag in (False, True):
    dl = make_train_loader(flag, batch_size=8)
    ds = dl.dataset
    x1, x2 = ds[0][0], ds[0][0]
    print(f"augment={flag!s:<5} same image twice -> max diff = {float((x1 - x2).abs().max()):.4f}")


raw training images: 50000 | used in this practice: 2000


augment=False same image twice -> max diff = 0.0000


augment=True  same image twice -> max diff = 3.4611


## Question 2 · Write the flip yourself

*torchvision.transforms.RandomHorizontalFlip* already handles both PIL images and tensors, but it
is a black box — write your own version so you can see the random branch and the flip.

Write a class *RandomHorizontalFlip(p=0.5)* whose *__call__(x)* takes a **tensor** of shape
*(B, C, H, W)* (or *(C, H, W)*) and

- flips it along the width with probability *p* (*torch.flip(x, [-1])*),
- returns it unchanged otherwise.

It is meant to sit *after* *ToTensor()*, so the same class also works on batches produced by the
loader. A probability outside *[0, 1]* raises *ValueError* in *__init__*, which is already
written; you fill in *__call__*.


In [2]:
# 🟢 EDIT: fill in __call__ of RandomHorizontalFlip.
# 🟡 READ: torch comes from the Q1 cell above.
class RandomHorizontalFlip:
    """Flip a tensor along the width axis with probability p."""

    # 🟢 CODE HERE - one method:
    #   __call__(self, x) : draw one random number with torch.rand(1).item();
    #        if it is smaller than self.p -> return torch.flip(x, [-1])   (the width axis)
    #        otherwise                   -> return x unchanged

    # 🟡 READ: __init__ is given - it checks the range and stores p
    def __init__(self, p=0.5):
        if not 0.0 <= p <= 1.0:
            raise ValueError("p must be in [0, 1]")
        self.p = p

    def __call__(self, x):
        if torch.rand(1).item() < self.p:
            return torch.flip(x, [-1])
        return x


# ⚪ RUN: p = 0 keeps the input, p = 1 flips it
x = torch.arange(2 * 3 * 4 * 4, dtype=torch.float32).view(2, 3, 4, 4)
for p in (0.0, 1.0):
    y = RandomHorizontalFlip(p)(x)
    print(f"p={p}: flipped = {not torch.equal(x, y)}")


p=0.0: flipped = False
p=1.0: flipped = True
